# Numerical mathematics with Convexity
## A worked problem book: undergraduate foundations to graduate numerical analysis

Solve problems, inspect diagnostics, and check the answers against exact mathematics. This notebook uses the **public Convexity interface** throughout; NumPy supplies input arrays and reference formulas. The final example shows how to bring your own library-backed tool.

**Run:** select your project virtual environment as the Jupyter kernel, then **Restart Kernel and Run All**. From the repository root, install `python -m pip install -e '.[notebooks,autodiff]'`. The `autodiff` extra is optional; without PyTorch, only its demonstration is skipped. No data downloads are needed. Figures and results are saved in this notebook.

| Level | Problems |
|---|---|
| Foundations | 1. Linear systems and factorizations · 2. Least squares and rank · 3. Roots · 4. Quadrature |
| Intermediate | 5. Derivatives · 6. Interpolation · 7. Conditioning · 8. Generalized eigenproblems and matrix equations |
| Advanced | 9. Sparse and matrix-free algebra · 10. Adaptive and stiff ODEs · 11. Boundary-value problems · 12. Discrete operators and mesh refinement |
| Extensibility | 13. Custom tools and explicit backend selection |

Each numerical check has a stated tolerance. Error estimates are algorithmic diagnostics, not guaranteed bounds. These examples demonstrate numerical mathematics only: no statistical estimation, optimization, stochastic models, or financial pricing.

In [ ]:
from pathlib import Path
import sys
import importlib.util
from importlib.metadata import version

# Works when launched from the repository root or notebooks.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "convexity" / "math" / "numerical").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from convexity.math import numerical as cx

plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True,
                     "grid.alpha": 0.25, "figure.dpi": 110})
np.set_printoptions(precision=7, suppress=True)
checks = []

def check(name, actual, expected, atol=1e-10, rtol=0.0):
    """Compare against a mathematical reference and retain an audit table."""
    actual, expected = np.broadcast_arrays(actual, expected)
    error = float(np.max(np.abs(actual - expected)))
    passed = cx.accuracy.allclose(actual, expected,
                                 tolerances=cx.Tolerances(atol=atol, rtol=rtol))
    checks.append(dict(check=name, max_abs_error=error, atol=atol, rtol=rtol, passed=passed))
    assert passed, f"{name}: max absolute error {error:.3g}"
    return error

display(pd.Series({p: version(p) for p in ("numpy", "scipy", "findiff", "matplotlib")},
                  name="Executed with"))

## 1. A linear system and four factorizations
Solve $Ax=b$ with an exact solution $x=(1,-2,3)^T$. Then reconstruct $A$ from LU, Cholesky, QR and Schur factors. QR with pivoting satisfies $A[:,p]=QR$; Schur satisfies $A=ZTZ^*$.

The returned `.value` is the answer; `.diagnostics` describes the solve. Prefer solving a system to explicitly forming an inverse.

In [ ]:
A = np.array([[4., 1., 0.], [1., 4., 1.], [0., 1., 3.]])
x_exact = np.array([1., -2., 3.])
b = A @ x_exact
sol = cx.linalg.solve(A, b)
check("dense solve", sol.value, x_exact)
lu = cx.linalg.lu(A).factors
chol = cx.linalg.cholesky(A).factors["l"]
qr = cx.linalg.qr(A, pivoting=True)
schur = cx.linalg.schur(A).factors
check("LU reconstruction", lu["p"] @ lu["l"] @ lu["u"], A)
check("Cholesky reconstruction", chol @ chol.T, A)
check("pivoted QR reconstruction", qr.factors["q"] @ qr.factors["r"], A[:, qr.permutation])
check("Schur reconstruction", schur["z"] @ schur["t"] @ schur["z"].conj().T, A)
print("x =", sol.value)
display(sol.diagnostics)

## 2. Overdetermined and rank-deficient systems
For an inconsistent system, the least-squares residual is orthogonal to the columns of $A$: $A^T(Ax-b)=0$. This is an algebraic projection, with no statistical model.

The second matrix has rank one. Its infinitely many exact solutions satisfy $x_1+2x_2=3$; the minimum-norm solution is $(3/5,6/5)$.

In [ ]:
C = np.array([[1., 0.], [0., 1.], [1., 1.]])
d = np.array([1., 2., 4.])
ls = cx.linalg.lstsq(C, d)
check("least-squares orthogonality", C.T @ (C @ ls.value - d), np.zeros(2))
R = np.array([[1., 2.], [2., 4.], [3., 6.]])
rhs = np.array([3., 6., 9.])
rank_deficient = cx.linalg.lstsq(R, rhs)
check("minimum-norm solution", rank_deficient.value, [.6, 1.2])
check("pseudoinverse solution", cx.linalg.pinv(R).value @ rhs, [.6, 1.2])
check("matrix rank", cx.linalg.matrix_rank(R).value, 1, atol=0)
svd = cx.linalg.svd(R)
check("SVD reconstruction", (svd.u * svd.s) @ svd.vh, R)
display(pd.DataFrame({"problem": ["inconsistent", "rank deficient"],
                      "rank": [ls.rank, rank_deficient.rank],
                      "residual norm": [ls.diagnostics.residual_norm,
                                        rank_deficient.diagnostics.residual_norm]}))

## 3. Scalar roots, nonlinear systems, and fixed points
Find the intersection of $y=\cos x$ and $y=x$ on $[0,1]$ using bracketing, Newton's method, and fixed-point iteration. Next solve $x^2+y^2=1$, $x=y$, selecting the positive solution via the initial guess.

An intentionally insufficient iteration budget demonstrates the failure contract: inspect a partial result with `check_convergence=False`, or catch `ConvergenceError` with the default behavior.

In [ ]:
f = lambda x: np.cos(x) - x
brent = cx.roots.root_scalar(f, bracket=(0., 1.))
newton = cx.roots.root_scalar(f, x0=.7, derivative=lambda x: -np.sin(x)-1,
                            tool=cx.ScalarRootSolver(method="newton"))
fixed = cx.roots.fixed_point(np.cos, .7)
for name, result in [("bracketing", brent), ("Newton", newton), ("fixed point", fixed)]:
    check(name + " residual", f(result.value), 0., atol=1e-9)
F = lambda z: np.array([z[0]**2 + z[1]**2 - 1, z[0] - z[1]])
system = cx.roots.solve(F, [.8, .6])
check("nonlinear system", system.value, np.ones(2)/np.sqrt(2), atol=1e-8)
print("Scalar root:", float(brent.value), "  System root:", system.value)
short = cx.ScalarRootSolver(method="bisect", maxiter=1)
partial = cx.roots.root_scalar(f, bracket=(0., 1.), tool=short, check_convergence=False)
assert not partial.diagnostics.converged
try:
    cx.roots.root_scalar(f, bracket=(0., 1.), tool=short)
except cx.ConvergenceError as exc:
    print("Expected failure:", exc)
else:
    raise AssertionError("Expected ConvergenceError")
display(partial.diagnostics)

## 4. Integrals with four different structures
Evaluate $\int_0^1\sqrt{x}\,dx=2/3$, the improper Gaussian integral $\int_{-\infty}^{\infty}e^{-x^2}dx=\sqrt\pi$, and $\int_0^1\int_0^1(x^2+y^2)\,dx\,dy=2/3$.

A four-node Gauss–Legendre rule integrates degree-seven polynomials exactly up to roundoff. Finally, integrate $x^2$ against the **supplied** weight $e^{-x}$ on $[0,\infty)$: the value is $2$. Convexity does not construct or normalize a probability distribution here.

In [ ]:
adaptive = cx.integrate.quad(np.sqrt, 0., 1.)
improper = cx.integrate.quad(lambda x: np.exp(-x*x), -np.inf, np.inf)
multi = cx.integrate.nquad(lambda z: z[0]**2 + z[1]**2, [(0., 1.), (0., 1.)])
gauss = cx.integrate.gauss_quad(lambda x: x**6, -1., 1., n=4)
weighted = cx.integrate.integrate_density(lambda x: x*x, lambda x: np.exp(-x), 0., np.inf)
rows = []
for name, result, exact in [("endpoint singular derivative", adaptive, 2/3),
                            ("improper Gaussian", improper, np.sqrt(np.pi)),
                            ("two-dimensional", multi, 2/3),
                            ("Gauss polynomial", gauss, 2/7),
                            ("supplied density", weighted, 2.)]:
    error = check(name, result.value, exact, atol=1e-9)
    rows.append({"integral": name, "value": float(result.value), "actual error": error,
                 "estimated error": result.diagnostics.error_estimate})
rule = cx.integrate.quadrature_rule(4, a=-1., b=1.)
check("quadrature nodes and weights", rule.weights @ rule.nodes**6, 2/7)
display(pd.DataFrame(rows))

## 5. Gradients, Hessians, Jacobians, and derivative products
For $f(x,y)=x^3+xy+y^4$, the exact gradient is $(3x^2+y,x+4y^3)$ and the Hessian is $\begin{pmatrix}6x&1\\1&12y^2\end{pmatrix}$.

For $F(x,y)=(xy,\sin x+y^2)$, verify the Jacobian and the identity $w^T(Jv)=(J^Tw)^Tv$. Finite differences work with NumPy callbacks. The optional PyTorch adapter uses tensor callbacks and returns the same Convexity result schema. This section differentiates functions; it does not optimize them.

In [ ]:
z = np.array([.7, -.4])
poly = lambda z: z[0]**3 + z[0]*z[1] + z[1]**4
g_exact = np.array([3*z[0]**2+z[1], z[0]+4*z[1]**3])
h_exact = np.array([[6*z[0], 1.], [1., 12*z[1]**2]])
g = cx.calculus.gradient(poly, z)
h = cx.calculus.hessian(poly, z)
check("finite-difference gradient", g.value, g_exact, atol=1e-7)
check("finite-difference Hessian", h.value, h_exact, atol=1e-6)
check("scalar derivative", cx.calculus.derivative(np.sin, .7).value, np.cos(.7), atol=1e-8)
vector_function = lambda z: np.array([z[0]*z[1], np.sin(z[0])+z[1]**2])
J_exact = np.array([[z[1], z[0]], [np.cos(z[0]), 2*z[1]]])
J = cx.calculus.jacobian(vector_function, z)
v, w = np.array([1., 2.]), np.array([-.5, 1.])
jv = cx.calculus.jvp(vector_function, z, v).value
jtw = cx.calculus.vjp(vector_function, z, w).value
check("Jacobian", J.value, J_exact, atol=1e-7)
check("JVP", jv, J_exact @ v, atol=1e-7)
check("VJP", jtw, J_exact.T @ w, atol=1e-7)
check("adjoint product identity", w @ jv, jtw @ v, atol=1e-7)
display(pd.DataFrame(h.value, columns=["d/dx", "d/dy"], index=["d/dx", "d/dy"]))

In [ ]:
torch_available = importlib.util.find_spec("torch") is not None
if torch_available:
    import torch
    torch_backend = cx.torch_backend()
    # Polynomial syntax works for both NumPy arrays and Torch tensors.
    ad_h = cx.calculus.hessian(poly, z, tool=torch_backend.tool("hessian"))
    check("PyTorch Hessian", ad_h.value, h_exact, atol=1e-12)
    derivative_check = cx.calculus.check_derivative(poly, z, ad_h, kind="hessian", atol=1e-6)
    assert derivative_check.passed
    def tensor_function(t):
        return torch.stack((t[0]*t[1], torch.sin(t[0])+t[1]**2))
    ad_jv = cx.calculus.jvp(tensor_function, z, v, tool=torch_backend.tool("jvp"))
    ad_vjp = cx.calculus.vjp(tensor_function, z, w, tool=torch_backend.tool("vjp"))
    check("PyTorch JVP", ad_jv.value, J_exact @ v, atol=1e-12)
    check("PyTorch VJP", ad_vjp.value, J_exact.T @ w, atol=1e-12)
    print("PyTorch", torch.__version__, "— automatic derivatives and finite-difference check passed")
else:
    print("OPTIONAL SECTION SKIPPED: install the autodiff extra to exercise PyTorch.")

## 6. The Runge phenomenon and multidimensional interpolation
Approximate $r(x)=1/(1+25x^2)$ on $[-1,1]$. High-degree polynomial interpolation on equally spaced points can oscillate badly near the endpoints. Compare it with Chebyshev–Lobatto nodes, a cubic spline, and shape-preserving PCHIP. A Chebyshev **basis alone** cannot fix poor sample locations.

Then interpolate the plane $g(x,y)=x+2y$ on a regular grid and on scattered points. Queries outside the supported domain raise by default; extrapolation is an explicit choice.

In [ ]:
runge = lambda x: 1/(1+25*x*x)
x_uniform = cx.grids.linspace(-1., 1., 17)
x_cheb = np.sort(np.cos(np.pi*np.arange(17)/16))
query = cx.grids.linspace(-1., 1., 1001)
curves = {
    "Uniform polynomial": cx.interpolation.polynomial_approximation(x_uniform, runge(x_uniform), 16),
    "Chebyshev nodes": cx.interpolation.polynomial_approximation(x_cheb, runge(x_cheb), 16),
    "Cubic spline": cx.interpolation.spline(x_uniform, runge(x_uniform)),
    "PCHIP": cx.interpolate(x_uniform, runge(x_uniform), tool=cx.Interpolation("pchip")),
}
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(query, runge(query), "k--", label="Exact")
errors = {}
for name, curve in curves.items():
    values = curve(query)
    errors[name] = np.max(np.abs(values-runge(query)))
    axes[0].plot(query, values, label=name)
    axes[1].semilogy(query, np.maximum(np.abs(values-runge(query)), 1e-16), label=name)
assert errors["Chebyshev nodes"] < errors["Uniform polynomial"]
check("Chebyshev interpolant at nodes", curves["Chebyshev nodes"](x_cheb), runge(x_cheb), atol=1e-9)
axes[0].set(xlabel="x", ylabel="Interpolated value", title="Sample placement matters")
axes[1].set(xlabel="x", ylabel="Absolute error", title="Error on a dense evaluation grid")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()
display(pd.Series(errors, name="Maximum sampled error"))

In [ ]:
axes_xy = (cx.grids.linspace(0., 1., 5), cx.grids.linspace(0., 1., 6))
X, Y = cx.grids.meshgrid(*axes_xy)
regular = cx.interpolation.regular_grid(axes_xy, X+2*Y)
points = np.array([[0., 0.], [1., 0.], [0., 1.], [1., 1.], [.4, .6]])
scattered = cx.interpolation.scattered(points, points[:, 0]+2*points[:, 1])
queries = np.array([[.2, .3], [.8, .7]])
exact = queries[:, 0] + 2*queries[:, 1]
check("regular-grid interpolation", regular(queries), exact)
check("scattered interpolation", scattered(queries), exact)
try:
    regular([[1.1, .5]])
except ValueError as exc:
    print("Expected domain validation:", exc)
else:
    raise AssertionError("Extrapolation must be explicit")
with_nan = cx.interpolation.regular_grid(axes_xy, X+2*Y, extrapolation="nan")
assert np.isnan(with_nan([[1.1, .5]])).all()
print("Explicit NaN extrapolation:", with_nan([[1.1, .5]]))

## 7. A tiny residual need not mean an accurate solution
The Hilbert matrix $H_{ij}=1/(i+j+1)$ is ill-conditioned. Start with $x=\mathbf1$, perturb $b=Hx$ by a relative $10^{-10}$ in the smallest left singular-vector direction, and solve again. The residual can remain tiny while the solution changes substantially.

Also inspect machine precision and evaluate a determinant in log space: a diagonal matrix with entries $10^{100}$ has log-determinant $4\times100\log10$, beyond ordinary determinant range.

In [ ]:
i = np.arange(8)
H = 1/(i[:, None]+i[None, :]+1.)
h_rhs = H @ np.ones(8)
h_svd = cx.linalg.svd(H)
delta = 1e-10 * cx.linalg.norm(h_rhs).value * h_svd.u[:, -1]
h_solution = cx.linalg.solve(H, h_rhs+delta)
h_residual = cx.accuracy.residual(H, h_solution.value, h_rhs+delta)
check("ill-conditioned solve residual", h_residual.relative, 0., atol=1e-14)
relative_change = np.linalg.norm(h_solution.value-1)/np.sqrt(8)
assert relative_change > 1e-3
logdet = cx.linalg.slogdet(np.eye(4)*1e100)
check("stable log-determinant", logdet.logabsdet, 400*np.log(10), atol=1e-10)
display(pd.Series({"condition number": float(cx.linalg.condition_number(H).value),
                  "relative RHS perturbation": np.linalg.norm(delta)/np.linalg.norm(h_rhs),
                  "relative solution change": relative_change,
                  "relative residual": h_residual.relative,
                  "float64 epsilon": cx.accuracy.machine_precision().epsilon,
                  "log absolute determinant": logdet.logabsdet}))
try:
    cx.accuracy.require_finite([1., np.nan])
except ValueError as exc:
    print("Expected finite-data validation:", exc)
else:
    raise AssertionError("NaN must be rejected")

## 8. Generalized modes, matrix functions, and matrix equations
For symmetric stiffness $K$ and positive-definite mass $M$, solve $Kv=\lambda Mv$. The eigenvectors are $M$-orthonormal. Next verify $\exp(tJ)$ for the rotation generator $J=\begin{pmatrix}0&-1\\1&0\end{pmatrix}$, plus matrix logarithm and square-root identities.

Finish with the continuous Lyapunov equation $AX+XA^T+Q=0$, its discrete counterpart $AXA^T-X+Q=0$, and scalar continuous/discrete algebraic Riccati equations. For $a=b=q=r=1$, their positive solutions are $1+\sqrt2$ and $(1+\sqrt5)/2$, respectively. These are matrix-equation examples without an application-specific control model.

In [ ]:
K = np.array([[2., -1.], [-1., 2.]])
M = np.diag([1., 2.])
modes = cx.linalg.eigh(K, M)
check("generalized eigenpair residual", K @ modes.vectors, (M @ modes.vectors)*modes.values)
check("mass orthogonality", modes.vectors.T @ M @ modes.vectors, np.eye(2))
angle = .7
J_rotation = np.array([[0., -1.], [1., 0.]])
rotation = cx.linalg.expm(angle*J_rotation).value
check("matrix exponential", rotation, [[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
check("matrix logarithm", cx.linalg.expm(cx.linalg.logm(K).value).value, K)
square_root = cx.linalg.sqrtm(K).value
check("matrix square root", square_root @ square_root, K)
for time, stable in [("continuous", np.diag([-1., -2.])), ("discrete", np.diag([.5, .7]))]:
    X_lyap = cx.linalg.solve_lyapunov(stable, np.eye(2), time=time).value
    residual = (stable @ X_lyap + X_lyap @ stable.T + np.eye(2) if time == "continuous"
                else stable @ X_lyap @ stable.T - X_lyap + np.eye(2))
    check(time + " Lyapunov equation", residual, np.zeros((2, 2)))
for time, exact in [("continuous", 1+np.sqrt(2)), ("discrete", (1+np.sqrt(5))/2)]:
    riccati = cx.linalg.solve_riccati([[1.]], [[1.]], [[1.]], [[1.]], time=time)
    check(time + " Riccati equation", riccati.value, [[exact]])
print("Generalized eigenvalues:", modes.values)
print("Matrix exponential:\n", rotation)

## 9. Sparse and matrix-free computation
Consider the $n\times n$ tridiagonal SPD matrix with diagonal $3$ and off-diagonals $-1$. Its eigenvalues are $\lambda_k=3-2\cos(k\pi/(n+1))$.

Construct a CSR representation, solve directly, then supply only the action $v\mapsto Av$ to conjugate gradients and GMRES. A diagonal inverse supplies a simple preconditioner. Sparse eigensolvers and SVD can compute only the required modes. No dense $n\times n$ matrix is formed.

In [ ]:
n = 200
indices = np.arange(n)
rows = np.r_[indices, indices[:-1], indices[1:]]
cols = np.r_[indices, indices[1:], indices[:-1]]
data = np.r_[np.full(n, 3.), -np.ones(n-1), -np.ones(n-1)]
sparse_A = cx.linalg.sparse_matrix((data, (rows, cols)), shape=(n, n))
def matvec(v):
    out = 3*v.copy()
    out[:-1] -= v[1:]
    out[1:] -= v[:-1]
    return out
operator = cx.linalg.linear_operator((n, n), matvec, rmatvec=matvec)
preconditioner = cx.linalg.linear_operator((n, n), lambda v: v/3, rmatvec=lambda v: v/3)
sparse_exact = np.sin(np.pi*(indices+1)/(n+1)) + .2*np.cos(.17*indices)
sparse_rhs = matvec(sparse_exact)
solver_rows = []
for name, matrix, tool in [("sparse direct", sparse_A, cx.SparseSolve()),
                            ("matrix-free CG", operator, cx.SparseSolve(method="cg", preconditioner=preconditioner)),
                            ("matrix-free GMRES", operator, cx.SparseSolve(method="gmres"))]:
    result = cx.linalg.solve(matrix, sparse_rhs, tool=tool)
    error = check(name, result.value, sparse_exact, atol=1e-8)
    solver_rows.append({"method": name, "max solution error": error,
                        "iterations": result.diagnostics.iterations,
                        "residual": result.diagnostics.residual_norm})
eigen = cx.linalg.sparse_eig(sparse_A, k=3, hermitian=True, which="SM")
check("sparse eigenvalues", np.sort(eigen.values), 3-2*np.cos(np.arange(1, 4)*np.pi/(n+1)), atol=1e-8)
sparse_svd = cx.linalg.sparse_svd(operator, k=2)
check("sparse singular values", sparse_svd.s, 3-2*np.cos(np.array([n, n-1])*np.pi/(n+1)), atol=1e-8)
check("sparse singular vector residual", sparse_A @ sparse_svd.vh.T, sparse_svd.u*sparse_svd.s, atol=1e-8)
display(pd.DataFrame(solver_rows))

## 10. Adaptive ODEs, events, dense output, and stiffness
First solve $q'=p$, $p'=-q$, with $(q(0),p(0))=(1,0)$. Its exact trajectory is $(\cos t,-\sin t)$. Locate downward crossings of $q=0$ at $\pi/2$ and $5\pi/2$, then evaluate the dense solution between output samples.

For stiffness, solve $y'=-1000(y-\cos t)-\sin t$, $y(0)=1$. The solution is $\cos t$, while perturbations decay on the much shorter timescale $10^{-3}$. Compare explicit RK45 with implicit Radau using the same requested tolerances. Function-evaluation counts are illustrative, not timing benchmarks.

In [ ]:
times = cx.grids.linspace(0., 4*np.pi, 160)
event = cx.ODEEvent(lambda t, y: y[0], direction=-1)
oscillator = cx.ode.solve(lambda t, y: np.array([y[1], -y[0]]), [1., 0.], times,
                         events=(event,), tool=cx.IVPSolver(method="DOP853", rtol=1e-9, atol=1e-11, max_step=.2))
fine_t = cx.grids.linspace(0., 4*np.pi, 1000)
dense = oscillator.solution(fine_t)
check("oscillator dense solution", dense, np.array([np.cos(fine_t), -np.sin(fine_t)]), atol=1e-8)
check("ODE event times", oscillator.events[0].t, [np.pi/2, 5*np.pi/2], atol=1e-8)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(fine_t, dense[0], label="Dense q(t)")
axes[0].scatter(oscillator.events[0].t, np.zeros(2), color="red", label="Downward zero crossing")
axes[0].set(xlabel="t", ylabel="q", title="Event detection"); axes[0].legend()
axes[1].plot(dense[0], dense[1]); axes[1].set(xlabel="q", ylabel="p", title="Phase trajectory", aspect="equal")
plt.tight_layout(); plt.show()

In [ ]:
stiff_rhs = lambda t, y: -1000*(y-np.cos(t))-np.sin(t)
stiff_t = cx.grids.linspace(0., 2., 201)
stiff_rows = []
for method in ("RK45", "Radau"):
    result = cx.ode.solve(stiff_rhs, [1.], stiff_t,
                          tool=cx.IVPSolver(method=method, atol=1e-9, rtol=1e-7),
                          jacobian=(lambda t, y: np.array([[-1000.]])) if method == "Radau" else None)
    error = check(method + " stiff equation", result.y[0], np.cos(stiff_t), atol=2e-6)
    stiff_rows.append({"method": method, "RHS evaluations": result.diagnostics.evaluations,
                       "max solution error": error})
display(pd.DataFrame(stiff_rows))

## 11. A two-point boundary-value problem
Solve $u''=-\pi^2\sin(\pi x)$ on $[0,1]$, with $u(0)=u(1)=0$. Set $y=(u,u')$ to obtain a first-order system. The exact solution is $u=\sin(\pi x)$.

Unlike an IVP callback, the BVP callback receives a **vector of coordinates** and an array of shape `(state, nodes)`. Start from zero on a coarse mesh; the solver adapts the mesh and supplies a continuous solution. Inspect both solution error and the solver's interval residual estimates.

In [ ]:
bvp_x = cx.grids.linspace(0., 1., 9)
def bvp_rhs(x, y):
    return np.vstack((y[1], -np.pi**2*np.sin(np.pi*x)))
boundary = lambda left, right: np.array([left[0], right[0]])
bvp = cx.ode.solve_bvp(bvp_rhs, boundary, bvp_x, np.zeros((2, len(bvp_x))))
eval_x = cx.grids.linspace(0., 1., 501)
bvp_values = bvp.solution(eval_x)
check("BVP solution", bvp_values[0], np.sin(np.pi*eval_x), atol=1e-7)
check("BVP derivative", bvp_values[1], np.pi*np.cos(np.pi*eval_x), atol=1e-7)
check("BVP boundary values", bvp.solution([0., 1.])[0], [0., 0.], atol=1e-10)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(eval_x, np.sin(np.pi*eval_x), "k--", label="Exact")
axes[0].plot(bvp.x, bvp.y[0], ".", label="Adaptive mesh")
axes[0].set(xlabel="x", ylabel="u", title=f"BVP: {len(bvp_x)} initial → {len(bvp.x)} final nodes")
axes[0].legend()
axes[1].semilogy(eval_x, np.maximum(np.abs(bvp_values[0]-np.sin(np.pi*eval_x)), 1e-16))
axes[1].set(xlabel="x", ylabel="Absolute error", title="Dense solution error")
plt.tight_layout(); plt.show()
print("Maximum interval RMS residual:", np.max(bvp.rms_residuals))

## 12. Discrete operators and convergence under mesh refinement
Differentiate $\sin x$ on the periodic interval $[0,2\pi)$ using fourth-order finite differences. Halving $h$ should reduce truncation error by about $2^4=16$ before roundoff dominates.

Then apply a two-dimensional periodic Laplacian to $u(x,y)=\sin x\cos y$, whose exact Laplacian is $-2u$. This illustrates reusable numerical primitives, without building a specialized PDE solver. Operators act on arrays flattened in **C order**. Periodic grids omit the duplicate endpoint.

In [ ]:
refinement = []
for count in (16, 32, 64, 128):
    grid = cx.grids.linspace(0., 2*np.pi, count, endpoint=False)
    D = cx.operators.differentiation_matrix(grid, order=1, accuracy=4, periodic=True)
    error = np.max(np.abs(D @ np.sin(grid)-np.cos(grid)))
    refinement.append({"nodes": count, "h": 2*np.pi/count, "max error": error})
refinement = pd.DataFrame(refinement)
refinement["observed order"] = np.log2(refinement["max error"].shift(1)/refinement["max error"])
assert (refinement["observed order"].dropna() > 3.9).all()
check("fourth-order derivative on finest grid", refinement["max error"].iloc[-1], 0., atol=2e-7)
display(refinement)
plt.loglog(refinement["h"], refinement["max error"], "o-", label="Measured error")
plt.loglog(refinement["h"], refinement["h"]**4/30, "--", label=r"$h^4/30$")
plt.xlabel("Grid spacing h"); plt.ylabel("Maximum derivative error"); plt.title("Mesh convergence")
plt.legend(); plt.show()

In [ ]:
axis = cx.grids.linspace(0., 2*np.pi, 64, endpoint=False)
XX, YY = cx.grids.meshgrid(axis, axis)
U = np.sin(XX)*np.cos(YY)
L = cx.operators.laplacian_matrix(U.shape, spacing=2*np.pi/64, boundary="periodic")
laplacian_U = (L @ U.ravel(order="C")).reshape(U.shape, order="C")
check("2D periodic Laplacian", laplacian_U, -2*U, atol=.002)
Dxy = cx.operators.partial_derivative_matrix((axis, axis), (1, 1), accuracy=4, periodic=True)
check("mixed partial derivative", (Dxy @ U.ravel()).reshape(U.shape),
      -np.cos(XX)*np.sin(YY), atol=1e-5)
products = cx.grids.cartesian_product([0., 1.], [10., 20., 30.])
check("Cartesian product ordering", products, [[0, 10], [0, 20], [0, 30], [1, 10], [1, 20], [1, 30]], atol=0)
print("Grid constructors:")
print("linear:", cx.grids.linspace(0, 1, 5))
print("logarithmic:", cx.grids.logspace(-3, 1, 5))
print("geometric:", cx.grids.geomspace(1, 16, 5))
print("range:", cx.grids.arange(0, 5, 1))
print("Laplacian shape:", L.shape, "stored entries:", L.nnz)

## 13. Bring your own tool; preserve the interface
A tool is a callable from a validated problem to a result. Here a custom dense solver delegates to `numpy.linalg.solve` and reports its measured residual. It plugs into both `tool=` and an immutable backend capability bundle. No inheritance, registry mutation, or private backend imports are required.

The interface checks input and output contracts; it cannot establish the mathematical correctness of an arbitrary custom algorithm. That remains the tool author's responsibility, which is why we check the result against the exact solution too.

In [ ]:
def numpy_solve_tool(problem: cx.LinearSystem) -> cx.ArrayResult:
    value = np.linalg.solve(problem.a, problem.b)
    residual_norm = float(np.linalg.norm(problem.a @ value - problem.b))
    return cx.ArrayResult(value, cx.Diagnostics(residual_norm=residual_norm,
                                               message="Custom tool using numpy.linalg.solve"))

custom = cx.linalg.solve(A, b, tool=numpy_solve_tool)
check("custom tool", custom.value, x_exact)
base_backend = cx.scipy_backend()
custom_backend = base_backend.with_tools(solve=numpy_solve_tool)
problem = cx.LinearSystem(A, b)
routed = custom_backend.run("solve", problem)
check("custom backend routing", routed.value, x_exact)
check("explicit run interface", cx.run(problem, numpy_solve_tool).value, x_exact)
assert base_backend.tool("solve") is not custom_backend.tool("solve")
assert not problem.b.flags.writeable
print(routed.diagnostics.message)
print("Available NumPy/SciPy backend capabilities:", ", ".join(sorted(base_backend.tools)))

# A wrong-shaped tool result is rejected at the interface boundary.
try:
    cx.run(problem, lambda p: cx.ArrayResult([0.]))
except ValueError as exc:
    print("Expected output-contract validation:", exc)
else:
    raise AssertionError("Invalid output shape must be rejected")

## Verification ledger and experiments to try
The table below records comparisons with exact solutions or defining identities. Additional assertions above check convergence failures, extrapolation, observed convergence order, finite-data handling, and custom-tool contracts. A skipped optional PyTorch section is explicitly reported.

To explore further, change one setting at a time and rerun the relevant section:
- Raise the Hilbert matrix size and watch conditioning grow.
- Change interpolation node counts and compare endpoint oscillations.
- Tighten ODE tolerances and inspect error and evaluation counts together.
- Change finite-difference accuracy from four to two and update the expected convergence order.

See [the numerical mathematics guide](../docs/numerical_mathematics.md) for the complete interface catalog and conventions.

In [ ]:
ledger = pd.DataFrame(checks)
assert ledger["passed"].all()
print(f"{len(ledger)} numerical reference checks passed.")
print("PyTorch coverage:", "executed" if torch_available else "skipped (optional dependency absent)")
print("All deliberate failures were caught and checked; no unhandled failures remain.")
with pd.option_context("display.max_rows", None, "display.float_format", "{:.3e}".format):
    display(ledger)